In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [3]:
import tensorflow as tf
import tqdm

In [64]:
class MultiHeadAttention(nn.Module):
  def __init__(self,context_length,embed_dim,d_out,num_heads,dropout,qkv_bias=False):
    super().__init__()
    self.d_out=d_out
    self.head_dim=d_out//num_heads
    self.num_heads=num_heads
    self.w_query=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.w_key=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.w_value=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.dropout=nn.Dropout(dropout)
    self.out_projection=nn.Linear(d_out,d_out)
    self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))
  def forward(self,x):
    b,num_tokens,embed_shape=x.shape
    query=self.w_query(x)
    key=self.w_key(x)
    value=self.w_value(x)       ### (b,num_tokens,d_out)###

    query=query.view(b,num_tokens,self.num_heads,self.head_dim)
    key=key.view(b,num_tokens,self.num_heads,self.head_dim)
    value=value.view(b,num_tokens,self.num_heads,self.head_dim)       ###(b,num_tokens,num_heads,head_dim)###

    query=query.transpose(1,2)
    key=key.transpose(1,2)
    value=value.transpose(1,2)          ###(b,num_heads,num_tokens,head_dim)

    attn_scores=query@key.transpose(2,3)
    attn_scores.masked_fill(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
    attn_weights=torch.softmax((attn_scores/(self.head_dim**0.5)),dim=-1)
    attn_weights=self.dropout(attn_weights)
    context_vec=attn_weights@value                ### (b,num_heads,num_tokens,num_tokens)@(b,num_heads,num_tokens,head_dim)

    context_vec=context_vec.transpose(1,2)                    ###(b,num_tokens,num_heads,head_dim)

    context_vec=context_vec.contiguous().view(b,num_tokens,self.d_out)
    context_vec=self.out_projection(context_vec)
    return context_vec

In [65]:
class LayerNorm(nn.Module):
  def __init__(self,embed_dim,eps=1e-5):
    super().__init__()
    self.eps=eps
    self.scale=nn.Parameter(torch.ones(embed_dim))
    self.shift=nn.Parameter(torch.zeros(embed_dim))
  def forward(self,x):
    mean=x.mean(dim=-1,keepdim=True)
    variance=x.var(dim=-1,unbiased=False,keepdim=True)
    norm_x=(x-mean)/torch.sqrt(variance+self.eps)
    return self.scale*norm_x + self.shift

In [66]:
class GeLU(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self,x):
    return nn.functional.gelu(x)

In [67]:
class FFNN(nn.Module):
  def __init__(self,embed_dim):
    super().__init__()
    self.network=nn.Sequential(
        nn.Linear(embed_dim,4*embed_dim),
        GeLU(),
        nn.Linear(4*embed_dim,embed_dim)
    )
  def forward(self,x):
    return self.network(x)

In [68]:
class TransformerBlock(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.attn=MultiHeadAttention(context_length=config["context_length"],embed_dim=config["embed_dim"],d_out=config["embed_dim"],num_heads=config["num_heads"],dropout=config["dropout"],qkv_bias=config["qkv_bias"])
    self.layer_norm1=LayerNorm(embed_dim=config["embed_dim"])
    self.layer_norm2=LayerNorm(embed_dim=config["embed_dim"])
    self.neural_net=FFNN(embed_dim=config["embed_dim"])
    self.dropout=nn.Dropout(config["dropout"])

  def forward(self,x):
    shortcut=x
    x=self.layer_norm1(x)
    x=self.attn(x)
    x=self.dropout(x)
    x=shortcut+x              ##Shortcut Connection
    shortcut=x
    x=self.layer_norm2(x)
    x=self.neural_net(x)
    x=self.dropout(x)
    x=shortcut+x              ###Shortcut Connection

    return x


In [69]:
class GPTModel(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.token_embedding=nn.Embedding(config["vocab_size"],config["embed_dim"])
    self.pos_embedding=nn.Embedding(config["context_length"],config["embed_dim"])
    self.dropout=nn.Dropout(config["dropout"])
    self.trf_blocks=nn.Sequential(*[TransformerBlock(config) for _ in range(config["n_layers"])])
    self.final_layer_norm=LayerNorm(config["embed_dim"])
    self.final_nn=nn.Linear(config["embed_dim"],config["vocab_size"],bias=False)
  def forward(self,x):
    batch,seq_length=x.shape
    token=self.token_embedding(x)
    pos=self.pos_embedding(torch.arange(seq_length,device=x.device))
    input_embedding=token+pos
    input_embedding=self.dropout(input_embedding)
    input_embedding=self.trf_blocks(input_embedding)
    input_embedding=self.final_layer_norm(input_embedding)
    logits=self.final_nn(input_embedding)
    return logits



In [70]:
class GPTDataset(Dataset):
  def __init__(self,document,tokenizer,context_length,stride):
    doc_tokens=tokenizer.encode(document,allowed_special={'<|endoftext|>'})
    self.input_ids=[]
    self.output_ids=[]
    for i in range(0,len(doc_tokens)-context_length,stride):
      input=doc_tokens[i:i+context_length]
      output=doc_tokens[i+1:i+1+context_length]
      self.input_ids.append(torch.tensor(input))
      self.output_ids.append(torch.tensor(output))
  def __len__(self):
    return len(self.input_ids)
  def __getitem__(self, index):
    return self.input_ids[index],self.output_ids[index]


In [71]:
def create_dataloader(document,context_length,stride,shuffle,drop_last,batch_size,num_workers):
  tokenizer=tiktoken.get_encoding('gpt2')
  dataset=GPTDataset(document=document,tokenizer=tokenizer,context_length=context_length,stride=stride)
  data_loader=DataLoader(dataset=dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)
  return data_loader

In [72]:
def calc_loss_total(data_loader,model,device,num_batches=None):
  total_loss=0
  if len(data_loader)==0:
    return float("nan")
  elif num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches,len(data_loader))
  for i,(input_batch,target_batch) in enumerate(data_loader):
    if i<num_batches:
      input_batch,target_batch=input_batch.to(device),target_batch.to(device)
      output_batch=model(input_batch)
      loss=torch.nn.functional.cross_entropy(output_batch.flatten(0,1),target_batch.flatten())
      total_loss+=loss.item()
    else:
      break
  return total_loss/num_batches




In [73]:
def text_to_token_ids(text,tokenizer):
  encoded_ids=tokenizer.encode(text,allowed_special={'<|endoftext|>'})
  return torch.tensor(encoded_ids).unsqueeze(0)

In [74]:
def token_ids_to_text(token_ids,tokenizer):
  flat = token_ids.squeeze(0)
  return tokenizer.decode(flat.tolist())

In [77]:
def train_model(model,train_loader,val_loader,optimizer,device,num_epochs,eval_freq,eval_iter):
  global_step=-1
  train_losses=[]
  val_losses=[]
  for i in range(num_epochs):
    model.train()
    for i,(input_batch,target_batch) in enumerate(train_loader):
      input_batch,target_batch=input_batch.to(device),target_batch.to(device)
      output_batch=model(input_batch)
      loss=torch.nn.functional.cross_entropy(output_batch.flatten(0,1),target_batch.flatten())
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      global_step+=1
      if global_step%(eval_freq)==0:
        with torch.no_grad():
            model.eval()
            train_loss=calc_loss_total(train_loader,model=model,device=device,num_batches=eval_iter)
            val_loss=calc_loss_total(val_loader,model=model,device=device,num_batches=eval_iter)
            model.train()
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            print(f"Ep {i+1} (step {global_step:06d}):" f" Train loss:{train_loss:.3f}, Val loss:{val_loss:.3f}")
  return train_losses,val_losses




In [90]:
def generate_new_text(model,context_length,max_new_tokens,input,top_k=None,temperature=0.0,eos_id=None):
  for _ in range(max_new_tokens):
    input_cond=input[:,-context_length:]      #(b,num_tokens)
    with torch.no_grad():
      logits=model(input_cond)           #(b,num_tokens,vocab_size)
    last_output_vec=logits[:,-1,:]
    if top_k is not None:
      top_logits,_=torch.topk(last_output_vec,top_k)
      masked_logits=torch.where(last_output_vec<top_logits[:,-1].unsqueeze(1),torch.tensor(float('-inf'),device=last_output_vec.device),last_output_vec)
    else:
      masked_logits=last_output_vec
    if temperature>0.0:
      masked_logits=masked_logits/temperature
      probs=torch.softmax(masked_logits,dim=-1)
      output_token=torch.multinomial(probs,num_samples=1)
    else:
      probs=torch.softmax(last_output_vec,dim=-1)    #(b,vocab_size)
      output_token=torch.argmax(probs,dim=-1,keepdim=True)      #(b,1)
    if output_token==eos_id:
      break
    input=torch.cat((input,output_token),dim=1)     #(b,num_tokens+1)
  return input

In [82]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    # Create a tensor from 'right' with the correct dtype and device
    new_data = torch.tensor(right, dtype=left.dtype, device=left.device)
    # Copy the new data into the existing parameter's data in-place
    left.data.copy_(new_data)
    return left

In [83]:
def load_weights_into_model(model,params):
  model.token_embedding.weight=assign(model.token_embedding.weight,params["wte"])
  model.pos_embedding.weight=assign(model.pos_embedding.weight,params["wpe"])
  for i in range(len(params["blocks"])):
    q_w,k_w,v_w=np.split(params['blocks'][i]['attn']['c_attn']['w'],3,axis=-1)
    model.trf_blocks[i].attn.w_query.weight=assign(model.trf_blocks[i].attn.w_query.weight,q_w.T)
    model.trf_blocks[i].attn.w_key.weight=assign(model.trf_blocks[i].attn.w_key.weight,k_w.T)
    model.trf_blocks[i].attn.w_value.weight=assign(model.trf_blocks[i].attn.w_value.weight,v_w.T)

    q_b,k_b,v_b=np.split(params['blocks'][i]['attn']['c_attn']['b'],3,axis=-1)
    model.trf_blocks[i].attn.w_query.bias=assign(model.trf_blocks[i].attn.w_query.bias,q_b)
    model.trf_blocks[i].attn.w_key.bias=assign(model.trf_blocks[i].attn.w_key.bias,k_b)
    model.trf_blocks[i].attn.w_value.bias=assign(model.trf_blocks[i].attn.w_value.bias,v_b)

    model.trf_blocks[i].attn.out_projection.weight = assign(
            model.trf_blocks[i].attn.out_projection.weight,
            params["blocks"][i]["attn"]["c_proj"]["w"].T)
    model.trf_blocks[b].attn.out_projection.bias = assign(
            model.trf_blocks[i].attn.out_projection.bias,
            params["blocks"][i]["attn"]["c_proj"]["b"])

    model.trf_blocks[i].neural_net.network[0].weight=assign(model.trf_blocks[i].neural_net.network[0].weight,params["blocks"][i]["mlp"]["c_fc"]["w"].T)
    model.trf_blocks[i].neural_net.network[0].bias=assign(model.trf_blocks[i].neural_net.network[0].bias,params["blocks"][i]["mlp"]["c_fc"]["b"])
    model.trf_blocks[i].neural_net.network[2].weight=assign(model.trf_blocks[i].neural_net.network[2].weight,params["blocks"][i]["mlp"]["c_proj"]["w"].T)
    model.trf_blocks[i].neural_net.network[2].bias=assign(model.trf_blocks[i].neural_net.network[2].bias,params["blocks"][i]["mlp"]["c_proj"]["b"])

    model.trf_blocks[i].layer_norm1.scale=assign(model.trf_blocks[i].layer_norm1.scale,params['blocks'][i]['ln_1']['g'])
    model.trf_blocks[i].layer_norm1.shift=assign(model.trf_blocks[i].layer_norm1.shift,params['blocks'][i]['ln_1']['b'])
    model.trf_blocks[i].layer_norm2.scale=assign(model.trf_blocks[i].layer_norm2.scale,params['blocks'][i]['ln_2']['g'])
    model.trf_blocks[i].layer_norm2.shift=assign(model.trf_blocks[i].layer_norm2.shift,params['blocks'][i]['ln_2']['b'])

  model.final_layer_norm.scale=assign(model.final_layer_norm.scale,params['g'])
  model.final_layer_norm.shift=assign(model.final_layer_norm.shift,params['b'])
  model.final_nn.weight=assign(model.final_nn.weight,params['wte'])



